In [8]:
tabla='cbtpa10'
dim='dim_paren'

In [9]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD_2"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [10]:

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM cbtpa10", con=connection2)

connection2.close()




In [11]:
base2

,tipoparecod,tipoparenom,tipopareequicod
0,01,TITULAR,01
1,02,CONYUGE,02
2,03,HIJO(A),03
3,04,HIJO(A) INCAPACITADO(A),None
4,05,CONCUBINO(A),None
5,06,GESTANTE,None
6,07,PADRE,13
7,08,MADRE,14
8,09,HERMANO(A),None
9,10,OTROS,None


In [12]:
base2.columns

Index(['tipoparecod', 'tipoparenom', 'tipopareequicod'], dtype='object')

In [13]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


query_count_before = "SELECT COUNT(*) FROM cbtpa10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla cbtpa10 antes de la inserción: {cant_antes}")


#connection3.execute('CREATE TEMPORARY TABLE tmp_cbtpa10 ()')
base2.to_sql(name='tmp_cbtpa10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cbtpa10 FALSO CON LO OBTENIDO DEL ORACLE

query="""

ALTER TABLE tmp_cbtpa10 
ALTER COLUMN tipoparecod TYPE character(2),
ALTER COLUMN tipoparenom TYPE character(30);



UPDATE cbtpa10 
SET 
tipoparecod=t.tipoparecod,
tipoparenom=t.tipoparenom

FROM tmp_cbtpa10 t 
WHERE cbtpa10.tipoparecod = t.tipoparecod and cbtpa10.tipoparecod IS NOT NULL;


INSERT INTO cbtpa10 
(tipoparecod,tipoparenom) 
SELECT 
tipoparecod,tipoparenom

FROM tmp_cbtpa10  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM cbtpa10 
    WHERE cbtpa10.tipoparecod = t.tipoparecod and cbtpa10.tipoparecod IS NOT NULL
) ;


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cbtpa10;
SELECT COUNT(*) FROM cbtpa10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla mbtae10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")
base2 = pd.read_sql_query(f"SELECT * FROM cbtpa10", con=connection3)









connection3.close()


Cantidad de filas en la tabla cbtpa10 antes de la inserción: 10
Cantidad de filas en la tabla mbtae10 después de la inserción: 10
La cantidad de filas insertadas fue de: 0


In [14]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN tipoparecod TYPE character(2),
ALTER COLUMN tipoparenom TYPE character(30);
INSERT INTO {dim} 

(cod_tpa,des_tpa) 

SELECT 


tipoparecod,tipoparenom

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d 
    WHERE 
    
    d.cod_tpa = t.tipoparecod
);
"""

c1= text(query)
cursor=connection4.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_paren antes de la inserción: 0
Cantidad de filas en la tabla dim_paren después de la inserción: 10
La cantidad de filas insertadas fue de: 10
